# Extracting structured data from PDFs — what we are building today

Before we take anything apart, let's watch it work.

Below, we point one function at a folder of PDFs, tell it what we want, and get
a table back. The folder holds two taxonomic papers: one born-digital from
2013, one a 1929 scan whose own text layer is quietly wrong in places.

The function works out which is which by itself, and tells you what it decided.

Nothing here is explained yet. That is the point — by the end of the day you
will have built every piece of it yourself, in `workshop.ipynb`, and you will
know exactly which parts you can trust.

**Set your runtime to a GPU first:** Runtime → Change runtime type → T4. Then
click *Connect*.

## Setup

This installs Ollama and downloads two models, which is about 14 GB. It is the
slowest thing we do all day — start it now and let it run while we talk.

In [ ]:
import glob, os, subprocess, sys, time

IN_COLAB = "google.colab" in sys.modules
REPO = "2026_UIUC_workshop_llm_pdf_extraction"

if IN_COLAB:
    # zstd unpacks Ollama's installer; pciutils lets it find the GPU.
    !DEBIAN_FRONTEND=noninteractive apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh
    !pip -q install ollama pymupdf pandas pillow
    if os.path.basename(os.getcwd()) != REPO:   # so this cell is safe to re-run
        if not os.path.isdir(REPO):
            !git clone -q https://github.com/de-Medeiros-insect-lab/{REPO}.git
        os.chdir(REPO)
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Jupyter puts the notebook's own folder on the import path, not the folder we
# are working in, so say explicitly where pdf_extraction.py lives.
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import ollama

def server_ready(timeout=120):
    """Wait until Ollama answers, or give up."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            ollama.list()
            return True
        except Exception:
            time.sleep(1)
    return False

assert server_ready(), "Ollama did not start"
print("Ollama is up")

In [ ]:
# ~6.5 GB: reads text and images, and follows a schema.
!ollama pull qwen3.5:9b
# ~6.7 GB: transcribes a page image. Only used on pages that need it.
!ollama pull deepseek-ocr

## The whole thing

Three things go in:

- **a folder** of PDFs,
- **a schema** — the fields you want, and their types,
- **a prompt** — what to extract, and what each field means.

One table comes out.

In [ ]:
from pdf_extraction import extract_folder

SCHEMA = {
    "type": "object",
    "properties": {
        "species": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name":       {"type": "string"},
                    "author":     {"type": "string"},
                    "min_length": {"type": "number"},
                    "max_length": {"type": "number"},
                },
                "required": ["name"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["species"],
    "additionalProperties": False,
}

PROMPT = (
    "List every species this paper describes. name is the species name. "
    "author is the taxonomic authority for that name -- the person who "
    "described it -- not an author of this paper. min_length and max_length "
    "are the smallest and largest body length in mm given for the species. "
    "Do not invent values that are not stated.\n\n"
)

df = extract_folder("example_pdfs", prompt=PROMPT, schema=SCHEMA)
df

## What just happened

Look at the progress lines. Before reading either paper, the function asked a
reasoning model one question about it — *can this document's own text be
trusted?* — and then printed what it decided and why.

The 2013 paper's text was fine, so reading it cost nothing. The 1929 paper has
a text layer too, and at a glance it looks plausible; it is the output of an OCR
pass from decades ago, and the model found spellings in it that are not words,
so every page was re-read from its image instead. Nothing inside a PDF tells
you which kind of document you have, and no rule about odd characters survives
the next scanner — which is exactly why that judgement is worth handing to a
model that can reason.

It found the figures in both papers and sent those too. Then it asked for the
fields in `SCHEMA` and nothing else, which is why the table has the columns it
has.

**Now check the table against the papers.** Somewhere in it there is probably
something wrong — a length that is really a ratio, an authority that is really
a phrase. The shape of an answer is guaranteed; its truth is not. That
distinction is most of what today is about.

## Your own PDFs

Same function, your folder. Upload a few PDFs with the folder icon 📁 in the
left sidebar, then edit the schema and prompt for what *you* want out of them.

Start with two or three documents. A vague prompt on four thousand PDFs is an
expensive way to learn that the prompt was vague.

Two arguments worth knowing about when you come back to this with real work:

- `needs_ocr=True` or `needs_ocr=False` skips the question and tells it
  outright. For your own material you usually know, and it saves a call per
  document.
- `cache_dir="somewhere"` writes each document's records as it finishes them
  and reads them back instead of redoing the work, so a long run that dies
  partway does not start over. Delete that folder when you change the prompt
  or the schema.

In [ ]:
os.makedirs("my_pdfs", exist_ok=True)

MY_SCHEMA = {
    "type": "object",
    "properties": {
        "records": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "FIELD_ONE": {"type": "string"},
                    "FIELD_TWO": {"type": "string"},
                },
                "required": ["FIELD_ONE"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["records"],
    "additionalProperties": False,
}

MY_PROMPT = (
    "DESCRIBE WHAT TO EXTRACT HERE. FIELD_ONE is ... FIELD_TWO is ... "
    "Do not invent values that are not stated.\n\n"
)

if not glob.glob("my_pdfs/*.pdf"):
    print("Upload PDFs into my_pdfs/ with the folder icon in the sidebar, "
          "then run this cell again.")
else:
    my_df = extract_folder("my_pdfs", prompt=MY_PROMPT, schema=MY_SCHEMA)
    my_df.to_csv("my_results.csv", index=False)
    display(my_df)

## Now let's take it apart

Everything you just ran lives in [`pdf_extraction.py`](pdf_extraction.py) — a
few hundred lines, yours to keep and to point at your own folders. You are
welcome to read it now, but it will not teach you much on its own: the
interesting parts are the decisions, not the code.

So: **close this notebook, disconnect the runtime, and open `workshop.ipynb`.**
We start from an empty cell and build up to what you just saw, in five
sessions:

1. **Setup** — talking to a language model from Python
2. **Preparing documents** — OCR and figure extraction
3. **Structured extraction** — getting tables out of unstructured text
4. **An agent** — letting the model choose its own approach
5. **Scaling up** — a bigger model in the cloud, and a whole folder of PDFs